# Module 7 — Day 1: Agent Fundamentals & Tool Design

**Goal:** Wrap Module 6 capabilities as tools and build a bounded ReAct-style agent loop.

Today: agent loops, tool contracts, tool registry, independent tool testing, tool-call tracing, and a small agent runner.

**Inputs:** `sample_multimodal_invoice.pdf`, `day4_retrieval_results.json`, and the ZIP of extracted Day 2 images. no GPU.

## 1. Install lightweight dependencies
The core exercise uses local PDF parsing and TF-IDF retrieval; no GPU or hosted LLM is required.

In [1]:
!pip -q install pymupdf pillow pandas scikit-learn

## 2. Imports and workspace
Create a shared workspace for uploaded Module 6 artifacts.

In [2]:
import json, re, shutil, zipfile
from pathlib import Path
from datetime import datetime
from typing import Any, Dict, List
import fitz, pandas as pd
from PIL import Image
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

WORK = Path("module7_day1")
IMG_DIR = WORK / "images"
IMG_DIR.mkdir(parents=True, exist_ok=True)
DOCUMENT_ELEMENTS, RETRIEVAL_RECORDS, TOOL_TRACE = [], [], []

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


## 3. Upload Module 6 artifacts
Upload the sample PDF, Day 4 retrieval JSON, and optionally the ZIP of extracted images. You can upload more than one file at once.

In [19]:
from google.colab import files
uploaded = files.upload()
pdfs = [n for n in uploaded if n.lower().endswith(".pdf")]
jsons = [n for n in uploaded if n.lower().endswith(".json")]
zips = [n for n in uploaded if n.lower().endswith(".zip")]
img_exts = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}

pdf_path = Path(pdfs[0]) if pdfs else None
retrieval_path = next((Path(n) for n in jsons if "day4_retrieval_results" in n.lower()), None)
for z in zips:
    with zipfile.ZipFile(z) as archive:
        archive.extractall(IMG_DIR)
for n in uploaded:
    if Path(n).suffix.lower() in img_exts:
        shutil.copy2(n, IMG_DIR / Path(n).name)
image_paths = sorted(p for p in IMG_DIR.rglob("*") if p.is_file() and p.suffix.lower() in img_exts)
print("PDF:", pdf_path.name if pdf_path else "not uploaded")
print("Retrieval JSON:", retrieval_path.name if retrieval_path else "not uploaded")
print("Images:", [p.name for p in image_paths])

Saving day2_extracted_images.zip to day2_extracted_images (1).zip
Saving day4_retrieval_results.json to day4_retrieval_results (1).json
Saving sample_multimodal_invoice.pdf to sample_multimodal_invoice.pdf
PDF: sample_multimodal_invoice.pdf
Retrieval JSON: day4_retrieval_results (1).json
Images: ['figure-1-1.jpg', 'table-1-1.jpg', 'table-1-2.jpg', 'table-2-3.jpg', 'table-2-4.jpg', 'table-2-5.jpg']


## 4. Shared state and trace logger
All tools return dictionaries with an `ok` field and preserve source metadata.

In [20]:
def log_tool(name, tool_input, observation):
    TOOL_TRACE.append({
        "step": len(TOOL_TRACE) + 1, "tool": name, "input": tool_input,
        "observation": observation, "timestamp": datetime.now().isoformat(timespec="seconds")
    })

## 5. Tool 1 — `parse_document`
Extract page-level text from the uploaded PDF. This is a lightweight wrapper; Module 6's Unstructured layout parser can replace it later.

In [21]:
def parse_document(file_path=None):
    path = Path(file_path) if file_path else pdf_path
    if path is None or not path.exists():
        return {"ok": False, "error": "Upload a PDF first."}
    elements = []
    with fitz.open(path) as doc:
        for page_no, page in enumerate(doc, 1):
            text = page.get_text("text").strip()
            if text:
                elements.append({
                    "source_id": f"PDF-P{page_no}", "modality": "text",
                    "source_doc": path.name, "page": page_no, "text": text,
                    "image_path": None
                })
    DOCUMENT_ELEMENTS.clear()
    DOCUMENT_ELEMENTS.extend(elements)
    result = {"ok": True, "document": path.name, "pages_with_text": len(elements),
              "preview": [e["text"][:220] for e in elements[:2]]}
    log_tool("parse_document", {"file_path": str(path)}, result)
    return result

## 6. Load Module 6 retrieval records
Normalize Day 4 JSON into a common schema so the agent tools can consume it.

In [22]:
def load_module6_results(path=retrieval_path):
    if path is None or not Path(path).exists():
        return {"ok": False, "message": "Day 4 retrieval JSON not uploaded."}
    with open(path, encoding="utf-8") as f:
        raw = json.load(f)
    records = raw if isinstance(raw, list) else raw.get("results", raw.get("retrieved_context", []))
    normalized = []
    for i, item in enumerate(records, 1):
        if not isinstance(item, dict):
            continue
        text = item.get("text") or item.get("content") or item.get("page_content") or ""
        if text:
            normalized.append({
                "source_id": item.get("source_id", f"M6-{i}"),
                "modality": item.get("modality", "text"),
                "source_doc": item.get("source_doc", item.get("source", "Module 6 result")),
                "page": item.get("page"), "text": str(text),
                "image_path": item.get("resolved_image_path") or item.get("image_path")
            })
    RETRIEVAL_RECORDS.clear()
    RETRIEVAL_RECORDS.extend(normalized)
    return {"ok": True, "records_loaded": len(normalized)}
print(load_module6_results())

{'ok': True, 'records_loaded': 0}


## 7. Tool 2 — `retrieve_multimodal`
This Day 1 retrieval wrapper uses TF-IDF over Module 6 text records (or parsed PDF pages). Later, replace the internals with the CLIP/FAISS index from Module 6.

In [23]:
def retrieve_multimodal(query, top_k=5):
    corpus = RETRIEVAL_RECORDS or DOCUMENT_ELEMENTS
    if not corpus:
        return {"ok": False, "error": "No records. Run parse_document or load Module 6 JSON."}
    texts = [r.get("text", "") for r in corpus]
    vectorizer = TfidfVectorizer(stop_words="english")
    matrix = vectorizer.fit_transform(texts + [query])
    scores = cosine_similarity(matrix[-1], matrix[:-1]).ravel()
    ranked = scores.argsort()[::-1][:top_k]
    results = [{
        "source_id": corpus[i].get("source_id"), "source_doc": corpus[i].get("source_doc"),
        "page": corpus[i].get("page"), "modality": corpus[i].get("modality", "text"),
        "score": round(float(scores[i]), 4), "text": corpus[i].get("text", "")[:1000]
    } for i in ranked]
    result = {"ok": True, "query": query, "results": results}
    log_tool("retrieve_multimodal", {"query": query, "top_k": top_k}, result)
    return result

## 8. Tool 3 — `search_metadata`
Use structured metadata filters when a query asks for a particular page, source document, or modality.

In [24]:
def search_metadata(source_doc=None, page=None, modality=None):
    corpus = RETRIEVAL_RECORDS or DOCUMENT_ELEMENTS
    matches = []
    for r in corpus:
        if source_doc and source_doc.lower() not in str(r.get("source_doc", "")).lower(): continue
        if page is not None and r.get("page") != page: continue
        if modality and modality.lower() != str(r.get("modality", "text")).lower(): continue
        matches.append({k: r.get(k) for k in ["source_id", "source_doc", "page", "modality"]})
    result = {"ok": True, "count": len(matches), "matches": matches[:20]}
    log_tool("search_metadata", {"source_doc": source_doc, "page": page, "modality": modality}, result)
    return result

## 9. Tool 4 — `caption_image`
For Day 1, return image metadata and a transparent placeholder caption. This lets us test the tool interface without loading a large VLM.

In [25]:
def caption_image(image_name):
    matches = [p for p in image_paths if p.name.lower() == Path(image_name).name.lower()]
    if not matches:
        return {"ok": False, "error": f"Image not found: {image_name}",
                "available_images": [p.name for p in image_paths]}
    path = matches[0]
    with Image.open(path) as im:
        width, height, fmt = im.width, im.height, im.format
    result = {"ok": True, "image": path.name,
              "caption": f"Extracted document image {path.name}, {width}x{height}px. Visual meaning requires a VLM.",
              "width": width, "height": height, "format": fmt, "image_path": str(path)}
    log_tool("caption_image", {"image_name": image_name}, result)
    return result

## 10. Tool 5 — `answer_vqa`
This is a Day 1 scaffold: it combines retrieved text with image metadata, but does not claim to understand image pixels. Connect Qwen3-VL in the later multimodal-query lesson.

In [26]:
def answer_vqa(question, image_name=None):
    image_info = caption_image(image_name) if image_name else {"ok": False, "message": "No image supplied."}
    text_evidence = retrieve_multimodal(question, top_k=3)
    result = {"ok": text_evidence.get("ok", False), "question": question,
              "image_info": image_info, "text_evidence": text_evidence.get("results", []),
              "limitation": "Scaffold only; not VLM-generated visual question answering."}
    log_tool("answer_vqa", {"question": question, "image_name": image_name}, result)
    return result

## 11. Register tools and inspect their contracts
The registry is the agent's tool catalogue: each tool has a name, description, and callable implementation.

In [27]:
TOOL_REGISTRY = {
    "parse_document": {"description": "Extract page-level text from an uploaded PDF.", "function": parse_document},
    "retrieve_multimodal": {"description": "Retrieve relevant text records for a query.", "function": retrieve_multimodal},
    "caption_image": {"description": "Return metadata for an extracted image.", "function": caption_image},
    "answer_vqa": {"description": "Combine text evidence and image metadata for a visual question.", "function": answer_vqa},
    "search_metadata": {"description": "Filter records by document, page, or modality.", "function": search_metadata}
}
for name, spec in TOOL_REGISTRY.items():
    print(name, "—", spec["description"])

parse_document — Extract page-level text from an uploaded PDF.
retrieve_multimodal — Retrieve relevant text records for a query.
caption_image — Return metadata for an extracted image.
answer_vqa — Combine text evidence and image metadata for a visual question.
search_metadata — Filter records by document, page, or modality.


## 12. Test tools independently
Always test tools before involving an agent. This separates retrieval or parsing bugs from planning errors.

In [28]:
if pdf_path and not RETRIEVAL_RECORDS:
    print(parse_document())
if RETRIEVAL_RECORDS or DOCUMENT_ELEMENTS:
    display(pd.DataFrame(retrieve_multimodal("invoice total supplier", top_k=3)["results"]))
    print(search_metadata(page=1))
else:
    print("Upload a PDF or Day 4 retrieval JSON to test retrieval.")

{'ok': True, 'document': 'sample_multimodal_invoice.pdf', 'pages_with_text': 2, 'preview': ['Synthetic sample document — for notebook testing\nPage 1\nTAX INVOICE\nOriginal for recipient\nSupplier\nMeridian Research Labs Pvt. Ltd.\nPlot 18, Sector 44\nGurugram, Haryana 122003\nGSTIN: 06AABCM1234F1Z5\nEmail: billing@merid', 'Synthetic sample document — for notebook testing\nPage 2\nService and Delivery Details\nSupporting schedule for invoice MRL-2026-1184\nDelivery Schedule\nDelivery Ref.\nItem\nDelivery Date\nLocation\nStatus\nDLV-8812\nWorkstations ']}


,source_id,source_doc,page,modality,score,text
0,PDF-P1,sample_multimodal_invoice.pdf,1,text,0.1156,Synthetic sample document — for notebook testi...
1,PDF-P2,sample_multimodal_invoice.pdf,2,text,0.0168,Synthetic sample document — for notebook testi...


{'ok': True, 'count': 1, 'matches': [{'source_id': 'PDF-P1', 'source_doc': 'sample_multimodal_invoice.pdf', 'page': 1, 'modality': 'text'}]}


## 13. Build a bounded ReAct-style loop
ReAct alternates between deciding what to do, calling a tool, observing its result, and deciding whether another action is needed. This first planner is deterministic so its choices are easy to inspect.

In [29]:
def choose_action(question, step):
    q = question.lower()
    if step == 1 and not (RETRIEVAL_RECORDS or DOCUMENT_ELEMENTS) and pdf_path:
        return {"tool": "parse_document", "args": {}}
    page_match = re.search(r"page\\s+(\\d+)", q)
    if page_match or "metadata" in q or "source id" in q:
        return {"tool": "search_metadata", "args": {"page": int(page_match.group(1)) if page_match else None}}
    if any(w in q for w in ["image", "diagram", "figure", "chart", "visual"]) and image_paths:
        return {"tool": "answer_vqa", "args": {"question": question, "image_name": image_paths[0].name}}
    return {"tool": "retrieve_multimodal", "args": {"query": question, "top_k": 4}}

def run_agent(question, max_steps=4):
    TOOL_TRACE.clear()
    steps = []
    for step in range(1, max_steps + 1):
        action = choose_action(question, step)
        observation = TOOL_REGISTRY[action["tool"]]["function"](**action["args"])
        steps.append({"step": step, "action": action, "observation": observation})
        if observation.get("ok") and action["tool"] in {"retrieve_multimodal", "answer_vqa", "search_metadata"}:
            break
    return {"question": question, "steps": steps, "max_steps": max_steps}
print("Agent loop ready; max_steps provides a hard stop.")

Agent loop ready; max_steps provides a hard stop.


## 14. Run a text-only agent query
The agent should choose retrieval and return evidence with source IDs and page metadata.

In [30]:
text_run = run_agent("What is the invoice total and who is the supplier?", max_steps=4)
print(json.dumps(text_run, indent=2)[:5000])

{
  "question": "What is the invoice total and who is the supplier?",
  "steps": [
    {
      "step": 1,
      "action": {
        "tool": "retrieve_multimodal",
        "args": {
          "query": "What is the invoice total and who is the supplier?",
          "top_k": 4
        }
      },
      "observation": {
        "ok": true,
        "query": "What is the invoice total and who is the supplier?",
        "results": [
          {
            "source_id": "PDF-P1",
            "source_doc": "sample_multimodal_invoice.pdf",
            "page": 1,
            "modality": "text",
            "score": 0.1156,
            "text": "Synthetic sample document \u2014 for notebook testing\nPage 1\nTAX INVOICE\nOriginal for recipient\nSupplier\nMeridian Research Labs Pvt. Ltd.\nPlot 18, Sector 44\nGurugram, Haryana 122003\nGSTIN: 06AABCM1234F1Z5\nEmail: billing@meridianlabs.example\nBill To\nNorthstar Office Supplies\nUnit 402, Tech Park\nJaipur, Rajasthan 302017\nGSTIN: 08AACCN5678L1Z2\nAt

## 15. Run a text-plus-image query
If extracted images were uploaded, the planner routes this query through the VQA scaffold. The output explicitly states that genuine visual interpretation is not yet connected.

In [31]:
if image_paths:
    image_run = run_agent("Explain the information shown in the invoice diagram.", max_steps=4)
    print(json.dumps(image_run, indent=2)[:5000])
else:
    print("Upload the Day 2 extracted-images ZIP to test image routing.")

{
  "question": "Explain the information shown in the invoice diagram.",
  "steps": [
    {
      "step": 1,
      "action": {
        "tool": "answer_vqa",
        "args": {
          "question": "Explain the information shown in the invoice diagram.",
          "image_name": "figure-1-1.jpg"
        }
      },
      "observation": {
        "ok": true,
        "question": "Explain the information shown in the invoice diagram.",
        "image_info": {
          "ok": true,
          "image": "figure-1-1.jpg",
          "caption": "Extracted document image figure-1-1.jpg, 896x262px. Visual meaning requires a VLM.",
          "width": 896,
          "height": 262,
          "format": "JPEG",
          "image_path": "module7_day1/images/figure-1-1.jpg"
        },
        "text_evidence": [
          {
            "source_id": "PDF-P1",
            "source_doc": "sample_multimodal_invoice.pdf",
            "page": 1,
            "modality": "text",
            "score": 0.043,
           

## 16. Save and inspect the tool-call trace
Tool I/O logging helps debug wrong tool choices, poor retrieval, and runaway loops.

In [32]:
trace_file = WORK / "day1_tool_trace.json"
with open(trace_file, "w", encoding="utf-8") as f:
    json.dump(TOOL_TRACE, f, indent=2, ensure_ascii=False)
display(pd.DataFrame([{"step": x["step"], "tool": x["tool"], "timestamp": x["timestamp"]} for x in TOOL_TRACE]))
print("Saved:", trace_file)

,step,tool,timestamp
0,1,caption_image,2026-11-19T14:17:04
1,2,retrieve_multimodal,2026-11-19T14:17:04
2,3,answer_vqa,2026-11-19T14:17:04


Saved: module7_day1/day1_tool_trace.json


## 17. Optional — inspect LangGraph
The explicit loop above teaches the mechanics. LangGraph will let us express the planner and tools as stateful graph nodes with conditional edges in the next lessons.

In [33]:
!pip -q install langgraph
from langgraph.graph import StateGraph, START, END
print("LangGraph ready:", StateGraph, START, END)

LangGraph ready: <class 'langgraph.graph.state.StateGraph'> __start__ __end__
